In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import time
import torch
import pandas as pd
import os

# import pygeohash as pgh

from collections import defaultdict
from torch_geometric.utils.convert import from_networkx
from torch_geometric.utils import to_undirected

from sklearn.model_selection import train_test_split

from tqdm.notebook import tqdm  # Use tqdm.notebook for Jupyter Notebook display

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import torch
import random

def set_seed(seed=42):
    """Set the same seed for reproducibility across NumPy, PyTorch, and Python."""
    np.random.seed(seed)             # NumPy
    torch.manual_seed(seed)          # PyTorch CPU
    torch.cuda.manual_seed(seed)     # PyTorch GPU (if applicable)
    torch.cuda.manual_seed_all(seed) # If using multiple GPUs
    random.seed(seed)                # Python's built-in random module

    # Ensure deterministic behavior in PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
SEED = 42

In [ ]:
set_seed(seed=SEED)

In [ ]:
base_path = '../data/processed'

In [ ]:
df = pd.read_csv(f'{base_path}/gowalla_all_events.csv', dtype={"uid": "Int64", "timestamp": "Int64", "other_uid": "Int64"})


In [ ]:
df

## Friend recommendation

In [ ]:
df_count = df.groupby(['uid', 'event_set']).agg(cnt=('event', 'count')).reset_index()
df_count

In [ ]:
df_freq = df_count.groupby(['event_set', 'cnt']).agg(freq=('uid', 'count')).reset_index()
df_freq

In [ ]:
dfr = df[df['event_set'] == 'relational'].reset_index()
dfr

In [ ]:
# 70-10-20 split, 0.125 * 0.8 = 0.1
dfr_train, dfr_test = train_test_split(dfr, test_size=0.2, random_state=SEED)
dfr_train, dfr_val = train_test_split(dfr_train, test_size=0.125, random_state=SEED)



In [ ]:
dfr_train = dfr_train.sort_values(['uid', 'other_uid'])[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfr_train

In [ ]:
dfr_test = dfr_test.sort_values(['uid', 'other_uid'])[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfr_test

In [ ]:
dfr_val = dfr_val.sort_values(['uid', 'other_uid'])[['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfr_val

In [ ]:
dfp = df[df['event_set'] == 'personal'].reset_index(drop=True)
dfp 

In [ ]:
task_path = '../data/tasks/gowalla/friend_recommendation'
os.makedirs(task_path, exist_ok=True)

dfp.to_csv(f'{task_path}/personal.csv', index=False)


In [ ]:
dfr_train.to_csv(f'{task_path}/relational_train.csv', index=False)
dfr_test.to_csv(f'{task_path}/relational_test.csv', index=False)
dfr_val.to_csv(f'{task_path}/relational_val.csv', index=False)

In [ ]:
## check
pd.read_csv(f'{task_path}/personal.csv')

### Negative sample

In [ ]:
NEG_SAMPLE_CNT = 50

In [ ]:
all_possible_ids = {int(uid) for uid in dfr['uid'].unique()}

def sample_negative(row, uid_to_positive, negative_count=10):
    """
    For a given row from df, sample negative ids.
    Negative samples are drawn from all_possible_ids excluding the positive ids
    for the given uid.
    """
    uid = row['uid']
    # Retrieve positive set for the uid, defaulting to empty set if not present.
    positive_set = uid_to_positive.get(uid, set())
    # Compute candidate negatives: all_possible_ids minus the positive set.
    candidates = list(all_possible_ids - positive_set)
    # Return negative_count samples, or fewer if not enough candidates.
    if len(candidates) < negative_count:
        return candidates
    else:
        return random.sample(candidates, negative_count)
    
    
def generate_negative_sample(dfr, uid_to_positive):
    negative_rows = []
    for idx, row in tqdm(dfr.iterrows(), total=len(dfr), desc="Generating negative samples"):
        neg_samples = sample_negative(row, uid_to_positive, negative_count=NEG_SAMPLE_CNT)
        for neg in neg_samples:
            negative_rows.append({
                'uid': row['uid'],
                'timestamp': row['timestamp'],
                'event_set': 'negative-' + row['event_set'],
                'event': row['event'],
                'other_uid': neg
            })

    # Create a new DataFrame from the negative sample rows.
    dfr_negative = pd.DataFrame(negative_rows)
    
    return dfr_negative
    


In [ ]:
# validation
dfr_val_pos = pd.concat([dfr_val, dfr_train]).groupby('uid')['other_uid'].agg(set).reset_index()
uid_to_positive = dict(zip(dfr_val_pos['uid'], dfr_val_pos['other_uid']))
dfr_val_neg = generate_negative_sample(dfr_val, uid_to_positive)
dfr_val_neg


In [ ]:
# test
dfr_test_pos = pd.concat([dfr_test, dfr_train]).groupby('uid')['other_uid'].agg(set).reset_index()
uid_to_positive = dict(zip(dfr_test_pos['uid'], dfr_test_pos['other_uid']))
dfr_test_neg = generate_negative_sample(dfr_test, uid_to_positive)
dfr_test_neg

In [ ]:
dfr_test_neg.to_csv(f'{task_path}/relational_test_negative_sample.csv', index=False)
dfr_val_neg.to_csv(f'{task_path}/relational_val_negative_sample.csv', index=False)

## Checkin Recommendation

In [ ]:
dfp

In [ ]:
dfp['order'] = dfp.groupby("uid").cumcount()
dfp['max_order'] = dfp.groupby("uid")['order'].transform('max')
dfp['rev_order'] = dfp['max_order'] - dfp['order']

dfp['order_prop'] = dfp['order'] / dfp['max_order']
dfp['rev_order_prop'] = dfp['rev_order'] / dfp['max_order']

In [ ]:
dfp['test_flag'] = ((dfp['rev_order_prop'] < 0.2) & (dfp['rev_order'] < 100))
dfp['val_flag'] = ((dfp['test_flag'] != True) & (dfp['rev_order_prop'] < 0.3) & (dfp['rev_order'] < 150))

In [ ]:
dfp

In [ ]:
dfp_train = dfp[((dfp['test_flag'] == False) & (dfp['val_flag'] == False))][['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfp_train


In [ ]:
dfp_val = dfp[(dfp['val_flag'] == True)][['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfp_val

In [ ]:
dfp_test = dfp[(dfp['test_flag'] == True)][['uid', 'timestamp', 'event_set', 'event', 'other_uid']].reset_index(drop=True)
dfp_test

In [ ]:
task_path = '../data/tasks/gowalla/checkin_prediction'
os.makedirs(task_path, exist_ok=True)

dfr.to_csv(f'{task_path}/relational.csv', index=False)


In [ ]:
dfp_train.to_csv(f'{task_path}/personal_train.csv', index=False)
dfp_test.to_csv(f'{task_path}/personal_test.csv', index=False)
dfp_val.to_csv(f'{task_path}/personal_val.csv', index=False)

### Negative sample

In [ ]:
NEG_SAMPLE_CNT = 10

In [ ]:
all_possible_ids = set(dfp['event'])

def sample_negative(row, uid_to_positive, negative_count=10):
    """
    For a given row from df, sample negative ids.
    Negative samples are drawn from all_possible_ids excluding the positive ids
    for the given uid.
    """
    uid = row['uid']
    # Retrieve positive set for the uid, defaulting to empty set if not present.
    positive_set = uid_to_positive.get(uid, set())
    # Compute candidate negatives: all_possible_ids minus the positive set.
    candidates = list(all_possible_ids - positive_set)
    # Return negative_count samples, or fewer if not enough candidates.
    if len(candidates) < negative_count:
        return candidates
    else:
        return random.sample(candidates, negative_count)
    
    
def generate_negative_sample(dfp, uid_to_positive):
    negative_rows = []
    for idx, row in tqdm(dfp.iterrows(), total=len(dfp), desc="Generating negative samples"):
        neg_samples = sample_negative(row, uid_to_positive, negative_count=NEG_SAMPLE_CNT)
        for neg in neg_samples:
            negative_rows.append({
                'uid': row['uid'],
                'timestamp': None,
                'event_set': 'negative-' +row['event_set'],
                'event': neg,
                'other_uid': None
            })

    # Create a new DataFrame from the negative sample rows.
    dfp_negative = pd.DataFrame(negative_rows)
    
    return dfp_negative

In [ ]:
# validation
dfp_val_pos = pd.concat([dfp_val, dfp_train]).groupby('uid')['event'].agg(set).reset_index()
uid_to_positive = dict(zip(dfp_val_pos['uid'], dfp_val_pos['event']))
dfp_val_neg = generate_negative_sample(dfp_val, uid_to_positive)
dfp_val_neg

In [ ]:
# test
dfp_test_pos = pd.concat([dfp_test, dfp_train]).groupby('uid')['event'].agg(set).reset_index()
uid_to_positive = dict(zip(dfp_test_pos['uid'], dfp_test_pos['event']))
dfp_test_neg = generate_negative_sample(dfp_test, uid_to_positive)
dfp_test_neg

In [ ]:
dfp_test_neg.to_csv(f'{task_path}/personal_test_negative_sample.csv', index=False)
dfp_val_neg.to_csv(f'{task_path}/personal_val_negative_sample.csv', index=False)